In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


class DecisionStump:
    """
    A weak learner: decision stump (single feature + threshold).
    Finds the best feature and threshold to minimise weighted error.
    """
    def __init__(self):
        self.feature_idx = None
        self.threshold = None
        self.polarity = 1      # 1 means: predict +1 if x >= threshold, -1 otherwise
        self.alpha = None

    def fit(self, X, y, sample_weights):
        """Train the stump using weighted samples."""
        n_samples, n_features = X.shape
        # y must be -1 or +1
        y = np.array(y)

        min_error = float('inf')
        best_feature = None
        best_threshold = None
        best_polarity = 1

        for feature in range(n_features):
            # Get unique values for this feature
            feature_values = X[:, feature]
            unique_vals = np.unique(feature_values)

            for threshold in unique_vals:
                for polarity in [1, -1]:
                    # Predictions: if polarity=1, predict +1 for x >= threshold
                    predictions = np.ones(n_samples)
                    if polarity == 1:
                        predictions[feature_values < threshold] = -1
                    else:
                        predictions[feature_values >= threshold] = -1

                    # Weighted error
                    incorrect = (predictions != y)
                    error = np.sum(sample_weights * incorrect)

                    if error < min_error:
                        min_error = error
                        best_feature = feature
                        best_threshold = threshold
                        best_polarity = polarity

        self.feature_idx = best_feature
        self.threshold = best_threshold
        self.polarity = best_polarity
        # alpha will be set by the boosting algorithm
        return self

    def predict(self, X):
        """Predict -1 or +1 for samples."""
        feature_vals = X[:, self.feature_idx]
        pred = np.ones(len(X))
        if self.polarity == 1:
            pred[feature_vals < self.threshold] = -1
        else:
            pred[feature_vals >= self.threshold] = -1
        return pred


class AdaBoost:
    """
    AdaBoost classifier using decision stumps as weak learners.
    Supports binary classification (y must be 0/1, internally mapped to -1/+1).
    """
    def __init__(self, n_estimators=50, learning_rate=1.0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.weak_learners = []      # list of (stump, alpha)
        self.classes_ = None

    def fit(self, X, y):
        """Train AdaBoost ensemble."""
        n_samples = X.shape[0]
        self.classes_ = np.unique(y)

        # Convert labels to -1 / +1
        y_encoded = np.where(y == self.classes_[0], -1, 1)

        # Initialise sample weights uniformly
        sample_weights = np.full(n_samples, 1.0 / n_samples)

        self.weak_learners = []

        for _ in range(self.n_estimators):
            # 1. Train a weighted decision stump
            stump = DecisionStump()
            stump.fit(X, y_encoded, sample_weights)

            # 2. Predict on training data
            predictions = stump.predict(X)

            # 3. Compute weighted error
            incorrect = (predictions != y_encoded)
            error = np.dot(sample_weights, incorrect) / np.sum(sample_weights)
            # Clip to avoid log(0) or division by zero
            error = np.clip(error, 1e-15, 1.0 - 1e-15)

            # 4. Compute learner weight (alpha)
            alpha = self.learning_rate * 0.5 * np.log((1 - error) / error)
            if alpha <= 0:
                break   # stop if no improvement

            # 5. Update sample weights
            # w_i <- w_i * exp(-alpha * y_i * h(x_i))
            sample_weights *= np.exp(-alpha * y_encoded * predictions)
            sample_weights /= np.sum(sample_weights)   # normalise

            # 6. Store the stump and its alpha
            stump.alpha = alpha
            self.weak_learners.append((stump, alpha))

        return self

    def predict(self, X):
        """Weighted majority vote."""
        if len(self.weak_learners) == 0:
            raise ValueError("Model not fitted or no weak learners were trained.")

        # Collect predictions from all stumps
        all_preds = np.array([stump.predict(X) for stump, _ in self.weak_learners])
        # Weighted sum: sum(alpha_m * h_m(x))
        alphas = np.array([alpha for _, alpha in self.weak_learners]).reshape(-1, 1)
        weighted_sum = np.sum(alphas * all_preds, axis=0)

        # Final prediction: sign of weighted sum
        pred_encoded = np.sign(weighted_sum)
        # Convert back to original class labels
        return np.where(pred_encoded == -1, self.classes_[0], self.classes_[1])

    def predict_proba(self, X):
        """Confidence‑based probability (sigmoid of weighted sum)."""
        all_preds = np.array([stump.predict(X) for stump, _ in self.weak_learners])
        alphas = np.array([alpha for _, alpha in self.weak_learners]).reshape(-1, 1)
        weighted_sum = np.sum(alphas * all_preds, axis=0)

        # Scaled sigmoid for probability of positive class (+1)
        prob_pos = 1.0 / (1.0 + np.exp(-2 * weighted_sum))
        prob_pos = np.clip(prob_pos, 1e-15, 1.0 - 1e-15)
        proba = np.column_stack((1 - prob_pos, prob_pos))
        return proba



# Example usage
if __name__ == "__main__":
    # Generate synthetic binary data
    X, y = make_classification(n_samples=300, n_features=10,
                               n_informative=8, n_redundant=2,
                               random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    # Train AdaBoost
    ada = AdaBoost(n_estimators=50, learning_rate=1.0)
    ada.fit(X_train, y_train)

    # Predict & evaluate
    y_pred = ada.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"AdaBoost Accuracy (pure NumPy): {acc:.3f}")   # ~0.944

    # Show probabilities
    proba = ada.predict_proba(X_test[:5])
    print("Predicted probabilities (first 5):\n", proba)

AdaBoost Accuracy (pure NumPy): 0.878
Predicted probabilities (first 5):
 [[0.88335532 0.11664468]
 [0.68847921 0.31152079]
 [0.18849205 0.81150795]
 [0.96852358 0.03147642]
 [0.96876086 0.03123914]]
